# <b><font color='forestgreen'>Polars: работа с несколькими таблицами</font></b>

### <font color='forestgreen'>1. Ответы на вопросы</font>

##### <font color='forestgreen'>1.1. Виды оконных функций в Polars</font>

In [2]:
import polars as pl

In [4]:
df_books = pl.read_csv('books_data_short.csv', null_values = 'NaN')
df_books

price,year,type
f64,f64,str
748.022957,1991.0,"""encyclopedia"""
5849.487399,null,"""magazine"""
546.485157,2021.0,"""book"""
1130.831881,1993.0,"""magazine"""
1247.822382,2002.0,"""magazine"""
1211.564183,2008.0,"""magazine"""
1651.054987,2009.0,"""newspaper"""
249.415919,1996.0,"""encyclopedia"""
605.61386,null,"""magazine"""


In [5]:
# Статистические функции (sum, min, max, median, std, count)
df_books.with_columns([
    pl.col("price").mean().over("type").alias("mean_price_by_type")
]).sort(by = ['type', "mean_price_by_type"])

price,year,type,mean_price_by_type
f64,f64,str,f64
546.485157,2021.0,"""book""",546.485157
748.022957,1991.0,"""encyclopedia""",498.719438
249.415919,1996.0,"""encyclopedia""",498.719438
5849.487399,null,"""magazine""",1902.560574
1130.831881,1993.0,"""magazine""",1902.560574
1247.822382,2002.0,"""magazine""",1902.560574
1211.564183,2008.0,"""magazine""",1902.560574
605.61386,null,"""magazine""",1902.560574
1370.04374,2008.0,"""magazine""",1902.560574


In [7]:
# Ранжирование - нумерация строк (rank)
# {‘average’, ‘min’, ‘max’, ‘dense’, ‘ordinal’, ‘random’} - default is ‘average’
# average: каждому элементу, входящему в группу одинаковых значений, присваивается среднее значение рангов, 
    # которые должны были быть назначены этим элементам.
# min: каждому элементу в группе одинаковых значений присваивается минимальный ранг из тех, которые могли быть назначены. 
# max: каждому элементу в группе одинаковых значений присваивается максимальный ранг из тех, которые могли быть назначены.
# dense: работает как min, но следующему элементу после группы одинаковых значений присваивается ранг, 
    # увеличенный только на 1 (без «пропусков» в нумерации).
# ordinal: всем элементам назначаются уникальные ранги в соответствии с порядком их появления в серии.
# random: как ordinal, но для одинаковых значений ранги назначаются случайным образом, не завися от порядка их появления.
df_books.with_columns([
    pl.col('year').rank('random', descending = False).over("type").alias("rank_year_by_type")
]).sort(by = ['type', 'year'])

price,year,type,rank_year_by_type
f64,f64,str,u32
546.485157,2021.0,"""book""",1
748.022957,1991.0,"""encyclopedia""",1
249.415919,1996.0,"""encyclopedia""",2
5849.487399,null,"""magazine""",null
605.61386,null,"""magazine""",null
1130.831881,1993.0,"""magazine""",1
1247.822382,2002.0,"""magazine""",2
1211.564183,2008.0,"""magazine""",4
1370.04374,2008.0,"""magazine""",3


In [8]:
# Разница между строками
df_books.with_columns([
    pl.col('price').diff(2).alias("diff_price")
])

price,year,type,diff_price
f64,f64,str,f64
748.022957,1991.0,"""encyclopedia""",null
5849.487399,null,"""magazine""",null
546.485157,2021.0,"""book""",-201.5378
1130.831881,1993.0,"""magazine""",-4718.655518
1247.822382,2002.0,"""magazine""",701.337225
1211.564183,2008.0,"""magazine""",80.732303
1651.054987,2009.0,"""newspaper""",403.232605
249.415919,1996.0,"""encyclopedia""",-962.148264
605.61386,null,"""magazine""",-1045.441127


In [9]:
# Значение следующей строки
df_books.with_columns([
    pl.col('price').shift(1).alias("previous_price")
])

price,year,type,previous_price
f64,f64,str,f64
748.022957,1991.0,"""encyclopedia""",null
5849.487399,null,"""magazine""",748.022957
546.485157,2021.0,"""book""",5849.487399
1130.831881,1993.0,"""magazine""",546.485157
1247.822382,2002.0,"""magazine""",1130.831881
1211.564183,2008.0,"""magazine""",1247.822382
1651.054987,2009.0,"""newspaper""",1211.564183
249.415919,1996.0,"""encyclopedia""",1651.054987
605.61386,null,"""magazine""",249.415919


In [10]:
# Значение предыдущей строки
df_books.with_columns([
    pl.col('price').shift(-1).alias("next_price")
])

price,year,type,next_price
f64,f64,str,f64
748.022957,1991.0,"""encyclopedia""",5849.487399
5849.487399,null,"""magazine""",546.485157
546.485157,2021.0,"""book""",1130.831881
1130.831881,1993.0,"""magazine""",1247.822382
1247.822382,2002.0,"""magazine""",1211.564183
1211.564183,2008.0,"""magazine""",1651.054987
1651.054987,2009.0,"""newspaper""",249.415919
249.415919,1996.0,"""encyclopedia""",605.61386
605.61386,null,"""magazine""",1370.04374


##### <font color='forestgreen'>1.2. Как с помощью select выбрать все колонки</font>

In [11]:
df_books.select('*',
               pl.when(pl.col('year') > 2014)
                .then(pl.lit(True))
                .otherwise(pl.lit(False))
                .alias('conditional'))

price,year,type,conditional
f64,f64,str,bool
748.022957,1991.0,"""encyclopedia""",false
5849.487399,null,"""magazine""",false
546.485157,2021.0,"""book""",true
1130.831881,1993.0,"""magazine""",false
1247.822382,2002.0,"""magazine""",false
1211.564183,2008.0,"""magazine""",false
1651.054987,2009.0,"""newspaper""",false
249.415919,1996.0,"""encyclopedia""",false
605.61386,null,"""magazine""",false


### <font color='forestgreen'>2. Join</font>

In [12]:
# Создадим 3 датафрейма в качестве примера
import numpy as np

np.random.seed(42)

# ID пользователя + имя
data1 = {
    'user_id': np.arange(1, 101),
    'name': np.random.choice(['Alice', 'Bob', 'Charlie', 'David'], 100)
}
df1 = pl.DataFrame(data1)

# ID пользователя + возраст
data2 = {
    'user_id': np.arange(51, 151),
    'age': np.random.randint(18, 65, size=100)
}
df2 = pl.DataFrame(data2)

# ID пользователя + оценка
data3 = {
    'user_id': np.random.randint(1, 101, size=100),
    'score': np.random.randint(200, 1000, size=100)
}
df3 = pl.DataFrame(data3)

In [34]:
df1

user_id,name
i64,str
1,"""Charlie"""
2,"""David"""
3,"""Alice"""
4,"""Charlie"""
5,"""Charlie"""
…,…
96,"""Bob"""
97,"""Bob"""
98,"""David"""


In [35]:
df2

user_id,age
i64,i32
51,35
52,43
53,61
54,51
55,27
…,…
146,39
147,45
148,19


In [36]:
df3

user_id,score
i32,i32
62,672
57,350
6,614
28,497
28,810
…,…
82,944
1,236
11,479


##### <font color='forestgreen'>2.1. Inner Join</font>

In [16]:
inner_joined_df = df1.join(df2, on = 'user_id', how = 'inner')
inner_joined_df

# Для сравнения та же операция в Pandas
# df1.join(df2, how = 'inner')
# pd.merge(df1, df1.join(df2, how = 'inner'),
#          left_on='user_id', right_on='user_id', how='inner')

user_id,name,age
i64,str,i32
51,"""Charlie""",35
52,"""Bob""",43
53,"""Charlie""",61
54,"""David""",51
55,"""Charlie""",27
…,…,…
96,"""Bob""",45
97,"""Bob""",24
98,"""David""",26


##### <font color='forestgreen'>2.2. Left Join</font>

In [17]:
# аналогично реализуется right join
left_joined_df = df1.join(df2, on='user_id', how='left')
left_joined_df

user_id,name,age
i64,str,i32
1,"""Charlie""",null
2,"""David""",null
3,"""Alice""",null
4,"""Charlie""",null
5,"""Charlie""",null
…,…,…
96,"""Bob""",45
97,"""Bob""",24
98,"""David""",26


##### <font color='forestgreen'>2.3. Full Join</font>

In [37]:
full_outer_joined_df = df1.join(df2, on='user_id', how='full')
full_outer_joined_df

user_id,name,user_id_right,age
i64,str,i64,i32
51,"""Charlie""",51,35
52,"""Bob""",52,43
53,"""Charlie""",53,61
54,"""David""",54,51
55,"""Charlie""",55,27
…,…,…,…
16,"""Alice""",null,null
40,"""Charlie""",null,null
24,"""David""",null,null


##### <font color='forestgreen'>2.4. Cross Join</font>

In [19]:
df_brands = pl.DataFrame({
    'brand': ['lacoste', 'prada', 'calvin klein']
})
df_sizes = pl.DataFrame({
    'sizes': ['S', 'M', 'L']
})
cross_joined_df = df_brands.join(df_sizes, how='cross')
cross_joined_df

brand,sizes
str,str
"""lacoste""","""S"""
"""lacoste""","""M"""
"""lacoste""","""L"""
"""prada""","""S"""
"""prada""","""M"""
"""prada""","""L"""
"""calvin klein""","""S"""
"""calvin klein""","""M"""
"""calvin klein""","""L"""


##### <font color='forestgreen'>2.5. Semi Join</font>
Возвращает все строки из левой таблицы, для которых есть совпадающие ключи в правой таблице, но не включает ни одного столбца из правой таблицы. Это эффективно фильтрует левую таблицу на основе наличия ключей в правой таблице.

In [20]:
semi_joined_df = df1.join(df2, on='user_id', how='semi')
semi_joined_df

# Аналог в Pandas
# df1[df1["id"].isin(df2["id"])]

user_id,name
i64,str
51,"""Charlie"""
52,"""Bob"""
53,"""Charlie"""
54,"""David"""
55,"""Charlie"""
…,…
96,"""Bob"""
97,"""Bob"""
98,"""David"""


##### <font color='forestgreen'>2.6. Anti Join</font>
Возвращает все строки из левой таблицы, для которых нет совпадающих ключей в правой таблице. Это полезно для поиска расхождений или исключений между двумя наборами данных.

In [21]:
anti_joined_df = df1.join(df2, on='user_id', how='anti')
anti_joined_df

# Аналог в Pandas
# df1[~df1["id"].isin(df2["id"])]

user_id,name
i64,str
1,"""Charlie"""
2,"""David"""
3,"""Alice"""
4,"""Charlie"""
5,"""Charlie"""
…,…
46,"""David"""
47,"""David"""
48,"""Charlie"""


### <font color='forestgreen'>3. Asof Join</font>

In [22]:
df_events = pl.DataFrame({
    "user_id": [1, 1, 2, 2],
    "event_time": ["2024-01-01 10:06", "2024-01-01 10:12", "2024-01-01 11:00", "2024-01-01 11:15"],
    "event_type": ['Добавить в корзину', 'Оплатить покупку', 'Открыть товар', 'Добавить в корзину']
}).with_columns(pl.col("event_time").str.strptime(pl.Datetime)).sort(['user_id','event_time'])
df_events

user_id,event_time,event_type
i64,datetime[μs],str
1,2024-01-01 10:06:00,"""Добавить в корзину"""
1,2024-01-01 10:12:00,"""Оплатить покупку"""
2,2024-01-01 11:00:00,"""Открыть товар"""
2,2024-01-01 11:15:00,"""Добавить в корзину"""


In [23]:
df_status = pl.DataFrame({
    "user_id": [1, 1, 2, 2],
    "status_time": ["2024-01-01 10:00", "2024-01-01 10:10", "2024-01-01 10:50", "2024-01-01 11:10"],
    "status_type": ['active', 'inactive', 'active', 'inactive']
}).with_columns(pl.col("status_time").str.strptime(pl.Datetime)).sort(['user_id','status_time'])
df_status

user_id,status_time,status_type
i64,datetime[μs],str
1,2024-01-01 10:00:00,"""active"""
1,2024-01-01 10:10:00,"""inactive"""
2,2024-01-01 10:50:00,"""active"""
2,2024-01-01 11:10:00,"""inactive"""


In [24]:
# Выполнение объединения по принципу asof - backward
asof_backward = df_events.join_asof(
    df_status,
    left_on = "event_time",
    right_on = "status_time",
    by = "user_id",      
    strategy = "backward"
)
asof_backward

# Аналог в Pandas
# asof_backward = pd.merge_asof(
#     events,      
#     status,
#     left_on = "event_time",
#     right_on = "status_time",
#     by = "user_id",                          
#     direction = "backward"                   
# )

C:\Users\Vitaliy\AppData\Local\Temp\ipykernel_8696\3845751449.py:2: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  asof_backward = df_events.join_asof(


user_id,event_time,event_type,status_time,status_type
i64,datetime[μs],str,datetime[μs],str
1,2024-01-01 10:06:00,"""Добавить в корзину""",2024-01-01 10:00:00,"""active"""
1,2024-01-01 10:12:00,"""Оплатить покупку""",2024-01-01 10:10:00,"""inactive"""
2,2024-01-01 11:00:00,"""Открыть товар""",2024-01-01 10:50:00,"""active"""
2,2024-01-01 11:15:00,"""Добавить в корзину""",2024-01-01 11:10:00,"""inactive"""


In [39]:
# Выполнение объединения по принципу asof - forward
asof_forward = df_events.join_asof(
    df_status,
    left_on = "event_time",
    right_on = "status_time",
    by = "user_id",      
    strategy = "forward"
)
asof_forward

C:\Users\Vitaliy\AppData\Local\Temp\ipykernel_8696\981689142.py:2: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  asof_forward = df_events.join_asof(


user_id,event_time,event_type,status_time,status_type
i64,datetime[μs],str,datetime[μs],str
1,2024-01-01 10:06:00,"""Добавить в корзину""",2024-01-01 10:10:00,"""inactive"""
1,2024-01-01 10:12:00,"""Оплатить покупку""",null,null
2,2024-01-01 11:00:00,"""Открыть товар""",2024-01-01 11:10:00,"""inactive"""
2,2024-01-01 11:15:00,"""Добавить в корзину""",null,null


In [26]:
# Выполнение объединения по принципу asof - nearest
asof_nearest = df_events.join_asof(
    df_status,
    left_on = "event_time",
    right_on = "status_time",
    by = "user_id",      
    strategy = "nearest" # при одинаковых разницах возьмет правое значение
)
asof_nearest

C:\Users\Vitaliy\AppData\Local\Temp\ipykernel_8696\4206014995.py:2: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  asof_nearest = df_events.join_asof(


user_id,event_time,event_type,status_time,status_type
i64,datetime[μs],str,datetime[μs],str
1,2024-01-01 10:06:00,"""Добавить в корзину""",2024-01-01 10:10:00,"""inactive"""
1,2024-01-01 10:12:00,"""Оплатить покупку""",2024-01-01 10:10:00,"""inactive"""
2,2024-01-01 11:00:00,"""Открыть товар""",2024-01-01 11:10:00,"""inactive"""
2,2024-01-01 11:15:00,"""Добавить в корзину""",2024-01-01 11:10:00,"""inactive"""


Параметр <b>tolerance</b>
- days
- seconds
- microseconds
- milliseconds
- minutes
- hours
- weeks

In [27]:
from datetime import timedelta

asof_backward = df_events.join_asof(
    df_status,
    left_on = "event_time",
    right_on = "status_time",
    by = "user_id",      
    strategy = "backward",
    tolerance = timedelta(seconds = 300)
)
asof_backward

C:\Users\Vitaliy\AppData\Local\Temp\ipykernel_8696\472331127.py:3: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  asof_backward = df_events.join_asof(


user_id,event_time,event_type,status_time,status_type
i64,datetime[μs],str,datetime[μs],str
1,2024-01-01 10:06:00,"""Добавить в корзину""",null,null
1,2024-01-01 10:12:00,"""Оплатить покупку""",2024-01-01 10:10:00,"""inactive"""
2,2024-01-01 11:00:00,"""Открыть товар""",null,null
2,2024-01-01 11:15:00,"""Добавить в корзину""",2024-01-01 11:10:00,"""inactive"""


### <font color='forestgreen'>4. Merge_sorted</font>

In [28]:
df11 = pl.DataFrame({
    "Name": ["John", "Joseph", "Albert"],
    "Age": [18, 15, 29]
}).sort("Age")

df12 = pl.DataFrame({
    "Name": ["Ema", "Andrew", "Michel"],
    "Age": [22, 30, 16]
}).sort("Age")

df13 = df11.merge_sorted(df12, "Age")
print(df13)

shape: (6, 2)
┌────────┬─────┐
│ Name   ┆ Age │
│ ---    ┆ --- │
│ str    ┆ i64 │
╞════════╪═════╡
│ Joseph ┆ 15  │
│ Michel ┆ 16  │
│ John   ┆ 18  │
│ Ema    ┆ 22  │
│ Albert ┆ 29  │
│ Andrew ┆ 30  │
└────────┴─────┘


### <font color='forestgreen'>5. Конкатенация</font>

##### <font color='forestgreen'>5.1. Вертикальная конкатенация</font>

In [29]:
df21 = pl.DataFrame({
    "Name": ["Alice", "Bob"],
    "Age": [25, 30]
})

df22 = pl.DataFrame({
    "Name": ["Charlie", "David"],
    "Age": [35, 40]
})

df_vertical = pl.concat([df21, df22], how = 'vertical')
df_vertical

# Аналог в pandas
# df_concat = pd.concat([df21, df22], axis=0, ignore_index=True)

Name,Age
str,i64
"""Alice""",25
"""Bob""",30
"""Charlie""",35
"""David""",40


##### <font color='forestgreen'>5.2. Горизонтальная конкатенация</font>

In [30]:
df23 = pl.DataFrame({
    "City": ["New York", "Los Angeles"],
    "Occupation": ["Engineer", "Doctor"]
})

df_horizontal = pl.concat([df21, df23], how="horizontal")
df_horizontal
# Аналог в pandas
# df_concat = pd.concat([df21, df23], axis=1, ignore_index=True)

Name,Age,City,Occupation
str,i64,str,str
"""Alice""",25,"""New York""","""Engineer"""
"""Bob""",30,"""Los Angeles""","""Doctor"""


##### <font color='forestgreen'>5.3. Диагональная конкатенация</font>

In [31]:
df_diagonal = pl.concat(
    [
        df21,
        df23,
    ],
    how = "diagonal"
)
print(df_diagonal)

shape: (4, 4)
┌───────┬──────┬─────────────┬────────────┐
│ Name  ┆ Age  ┆ City        ┆ Occupation │
│ ---   ┆ ---  ┆ ---         ┆ ---        │
│ str   ┆ i64  ┆ str         ┆ str        │
╞═══════╪══════╪═════════════╪════════════╡
│ Alice ┆ 25   ┆ null        ┆ null       │
│ Bob   ┆ 30   ┆ null        ┆ null       │
│ null  ┆ null ┆ New York    ┆ Engineer   │
│ null  ┆ null ┆ Los Angeles ┆ Doctor     │
└───────┴──────┴─────────────┴────────────┘


### <font color='forestgreen'>7. Unpivot</font>

In [32]:
df = pl.DataFrame({
"Year": [2020, 2021],
"Product A": [100, 150],
"Product B": [90, 120]
})
print(df)

melted_df = df.unpivot(
                index = "Year", # Колонка, которая остается как есть
                on = ["Product A", "Product B"], # колонки, которые превращаются в строки
                variable_name = "Product", # имя новой колонки для исходных названий столбцов
                value_name = "Sales") # имя новой колонки для значений
print(melted_df)

shape: (2, 3)
┌──────┬───────────┬───────────┐
│ Year ┆ Product A ┆ Product B │
│ ---  ┆ ---       ┆ ---       │
│ i64  ┆ i64       ┆ i64       │
╞══════╪═══════════╪═══════════╡
│ 2020 ┆ 100       ┆ 90        │
│ 2021 ┆ 150       ┆ 120       │
└──────┴───────────┴───────────┘
shape: (4, 3)
┌──────┬───────────┬───────┐
│ Year ┆ Product   ┆ Sales │
│ ---  ┆ ---       ┆ ---   │
│ i64  ┆ str       ┆ i64   │
╞══════╪═══════════╪═══════╡
│ 2020 ┆ Product A ┆ 100   │
│ 2021 ┆ Product A ┆ 150   │
│ 2020 ┆ Product B ┆ 90    │
│ 2021 ┆ Product B ┆ 120   │
└──────┴───────────┴───────┘


In [40]:
import pandas as pd

# Аналог в pandas
df = pd.DataFrame({
"Year": [2020, 2021],
"Product A": [100, 150],
"Product B": [90, 120]
})

df_long = df.melt(
    id_vars = "Year", 
    value_vars = ["Product A", "Product B"], 
    var_name = "Product",
    value_name = "Sales"
)
print(df_long)

   Year    Product  Sales
0  2020  Product A    100
1  2021  Product A    150
2  2020  Product B     90
3  2021  Product B    120


### <font color='forestgreen'>8. Задача "Анализ продаж и активности клиентов"</font>

<b> Описание данных: </b>
- <i>Таблица users</i> (client_id, name, city) - информация о пользователях
- <i>Таблица events</i> (user_id, event_time, event_type) - события пользователей
- <i>Таблица sales</i> (order_id, user_id, product, amount, order_time) - продажи товаров
- <i>Таблица inventory</i> (product, stock_jan, stock_feb, stock_march) - остатки товаров по месяцам

In [41]:
from datetime import timedelta
# Генерация данных
users = pl.DataFrame({
    "user_id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "city": ["New York", "Los Angeles", "Chicago"]
})

events = pl.DataFrame({
    "user_id": [1, 1, 2, 3],
    "event_time": ["2025-09-01 10:05", "2025-09-01 10:12", "2025-09-01 11:00", "2025-09-01 12:00"],
    "event_type": ["click", "purchase", "click", "click"]
}).with_columns(pl.col("event_time").str.strptime(pl.Datetime))

sales = pl.DataFrame({
    "order_id": [1001, 1002, 1003],
    "user_id": [1, 2, 4], 
    "product": ["Shoes", "Shirt", "Hat"],
    "amount": [120, 40, 25],
    "order_time": ["2025-09-01 10:12", "2025-09-01 11:05", "2025-09-01 12:10"]
}).with_columns(pl.col("order_time").str.strptime(pl.Datetime))

inventory = pl.DataFrame({
    "product": ["Shoes", "Shirt", "Hat"],
    "stock_jan": [50, 100, 20],
    "stock_feb": [45, 90, 15],
    "stock_mar": [40, 80, 10]
})

In [44]:
# Выведите информацию только о тех пользователях, которые сделали покупку
print(users)
print(sales)
user_sales = users.join(sales, on='user_id', how='semi')
display(user_sales)

shape: (3, 3)
┌─────────┬─────────┬─────────────┐
│ user_id ┆ name    ┆ city        │
│ ---     ┆ ---     ┆ ---         │
│ i64     ┆ str     ┆ str         │
╞═════════╪═════════╪═════════════╡
│ 1       ┆ Alice   ┆ New York    │
│ 2       ┆ Bob     ┆ Los Angeles │
│ 3       ┆ Charlie ┆ Chicago     │
└─────────┴─────────┴─────────────┘
shape: (3, 5)
┌──────────┬─────────┬─────────┬────────┬─────────────────────┐
│ order_id ┆ user_id ┆ product ┆ amount ┆ order_time          │
│ ---      ┆ ---     ┆ ---     ┆ ---    ┆ ---                 │
│ i64      ┆ i64     ┆ str     ┆ i64    ┆ datetime[μs]        │
╞══════════╪═════════╪═════════╪════════╪═════════════════════╡
│ 1001     ┆ 1       ┆ Shoes   ┆ 120    ┆ 2025-09-01 10:12:00 │
│ 1002     ┆ 2       ┆ Shirt   ┆ 40     ┆ 2025-09-01 11:05:00 │
│ 1003     ┆ 4       ┆ Hat     ┆ 25     ┆ 2025-09-01 12:10:00 │
└──────────┴─────────┴─────────┴────────┴─────────────────────┘


user_id,name,city
i64,str,str
1,"""Alice""","""New York"""
2,"""Bob""","""Los Angeles"""


In [45]:
# Выведите информацию о всех пользователях + информацию о покупке, если она есть
user_total_sales = users.join(sales, on='user_id', how='left')
display(user_total_sales)

user_id,name,city,order_id,product,amount,order_time
i64,str,str,i64,str,i64,datetime[μs]
1,"""Alice""","""New York""",1001,"""Shoes""",120,2025-09-01 10:12:00
2,"""Bob""","""Los Angeles""",1002,"""Shirt""",40,2025-09-01 11:05:00
3,"""Charlie""","""Chicago""",null,null,null,null


In [46]:
# Выведите информацию о всех пользователях и всех заказах (включая заказы от пользователей, которых нет в users).
all_user_sales = users.join(sales, on='user_id', how='full')
display(all_user_sales)

user_id,name,city,order_id,user_id_right,product,amount,order_time
i64,str,str,i64,i64,str,i64,datetime[μs]
1,"""Alice""","""New York""",1001,1,"""Shoes""",120,2025-09-01 10:12:00
2,"""Bob""","""Los Angeles""",1002,2,"""Shirt""",40,2025-09-01 11:05:00
null,null,null,1003,4,"""Hat""",25,2025-09-01 12:10:00
3,"""Charlie""","""Chicago""",null,null,null,null,null


In [47]:
# Создайте все возможные пары пользователь × продукт, чтобы посчитать потенциальные покупки.
print(users)
print(inventory)
potential_purchases = users.join(inventory, how='cross')
display(potential_purchases)

shape: (3, 3)
┌─────────┬─────────┬─────────────┐
│ user_id ┆ name    ┆ city        │
│ ---     ┆ ---     ┆ ---         │
│ i64     ┆ str     ┆ str         │
╞═════════╪═════════╪═════════════╡
│ 1       ┆ Alice   ┆ New York    │
│ 2       ┆ Bob     ┆ Los Angeles │
│ 3       ┆ Charlie ┆ Chicago     │
└─────────┴─────────┴─────────────┘
shape: (3, 4)
┌─────────┬───────────┬───────────┬───────────┐
│ product ┆ stock_jan ┆ stock_feb ┆ stock_mar │
│ ---     ┆ ---       ┆ ---       ┆ ---       │
│ str     ┆ i64       ┆ i64       ┆ i64       │
╞═════════╪═══════════╪═══════════╪═══════════╡
│ Shoes   ┆ 50        ┆ 45        ┆ 40        │
│ Shirt   ┆ 100       ┆ 90        ┆ 80        │
│ Hat     ┆ 20        ┆ 15        ┆ 10        │
└─────────┴───────────┴───────────┴───────────┘


user_id,name,city,product,stock_jan,stock_feb,stock_mar
i64,str,str,str,i64,i64,i64
1,"""Alice""","""New York""","""Shoes""",50,45,40
1,"""Alice""","""New York""","""Shirt""",100,90,80
1,"""Alice""","""New York""","""Hat""",20,15,10
2,"""Bob""","""Los Angeles""","""Shoes""",50,45,40
2,"""Bob""","""Los Angeles""","""Shirt""",100,90,80
2,"""Bob""","""Los Angeles""","""Hat""",20,15,10
3,"""Charlie""","""Chicago""","""Shoes""",50,45,40
3,"""Charlie""","""Chicago""","""Shirt""",100,90,80
3,"""Charlie""","""Chicago""","""Hat""",20,15,10


In [48]:
# Выберите пользователей, которые не делали ни одного заказа.
not_purchased_users = users.join(sales, on='user_id', how='anti')
display(not_purchased_users)

user_id,name,city
i64,str,str
3,"""Charlie""","""Chicago"""


In [60]:
# Найдите для каждого события пользователя (events) ближайший по времени заказ из sales.
# 1. Отсортируйте заказы по времени
print(events)
print(sales)
event_sales = events.join_asof(
    sales,
    left_on = "event_time",
    right_on = "order_time",
    by = "user_id",
    strategy = "nearest"
).sort(by = ['user_id', 'event_time'])
display(event_sales)

# 2. Примените asof join
# Используйте strategy = backward и tolerance = 30 минут
event_sales_backward = events.join_asof(
    sales,
    left_on = "event_time",
    right_on = "order_time",
    by = "user_id",
    strategy = "backward",
    tolerance = timedelta(minutes = 30)
)
display(event_sales_backward)

shape: (4, 3)
┌─────────┬─────────────────────┬────────────┐
│ user_id ┆ event_time          ┆ event_type │
│ ---     ┆ ---                 ┆ ---        │
│ i64     ┆ datetime[μs]        ┆ str        │
╞═════════╪═════════════════════╪════════════╡
│ 1       ┆ 2025-09-01 10:05:00 ┆ click      │
│ 1       ┆ 2025-09-01 10:12:00 ┆ purchase   │
│ 2       ┆ 2025-09-01 11:00:00 ┆ click      │
│ 3       ┆ 2025-09-01 12:00:00 ┆ click      │
└─────────┴─────────────────────┴────────────┘
shape: (3, 5)
┌──────────┬─────────┬─────────┬────────┬─────────────────────┐
│ order_id ┆ user_id ┆ product ┆ amount ┆ order_time          │
│ ---      ┆ ---     ┆ ---     ┆ ---    ┆ ---                 │
│ i64      ┆ i64     ┆ str     ┆ i64    ┆ datetime[μs]        │
╞══════════╪═════════╪═════════╪════════╪═════════════════════╡
│ 1001     ┆ 1       ┆ Shoes   ┆ 120    ┆ 2025-09-01 10:12:00 │
│ 1002     ┆ 2       ┆ Shirt   ┆ 40     ┆ 2025-09-01 11:05:00 │
│ 1003     ┆ 4       ┆ Hat     ┆ 25     ┆ 2025-09-01 1

C:\Users\Vitaliy\AppData\Local\Temp\ipykernel_8696\2829909152.py:5: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  event_sales = events.join_asof(


user_id,event_time,event_type,order_id,product,amount,order_time
i64,datetime[μs],str,i64,str,i64,datetime[μs]
1,2025-09-01 10:05:00,"""click""",1001,"""Shoes""",120,2025-09-01 10:12:00
1,2025-09-01 10:12:00,"""purchase""",1001,"""Shoes""",120,2025-09-01 10:12:00
2,2025-09-01 11:00:00,"""click""",1002,"""Shirt""",40,2025-09-01 11:05:00
3,2025-09-01 12:00:00,"""click""",null,null,null,null


C:\Users\Vitaliy\AppData\Local\Temp\ipykernel_8696\2829909152.py:16: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  event_sales_backward = events.join_asof(


user_id,event_time,event_type,order_id,product,amount,order_time
i64,datetime[μs],str,i64,str,i64,datetime[μs]
1,2025-09-01 10:05:00,"""click""",null,null,null,null
1,2025-09-01 10:12:00,"""purchase""",1001,"""Shoes""",120,2025-09-01 10:12:00
2,2025-09-01 11:00:00,"""click""",null,null,null,null
3,2025-09-01 12:00:00,"""click""",null,null,null,null


In [57]:
# Объедините несколько DataFrame с остатками товаров (stock_jan, stock_feb, stock_mar) в один DataFrame, 
# где строки идут друг за другом.
print(inventory)
# print(sales)
inventory_vertical = pl.concat([inventory,inventory, inventory], how = 'vertical')
display(inventory_vertical)

shape: (3, 4)
┌─────────┬───────────┬───────────┬───────────┐
│ product ┆ stock_jan ┆ stock_feb ┆ stock_mar │
│ ---     ┆ ---       ┆ ---       ┆ ---       │
│ str     ┆ i64       ┆ i64       ┆ i64       │
╞═════════╪═══════════╪═══════════╪═══════════╡
│ Shoes   ┆ 50        ┆ 45        ┆ 40        │
│ Shirt   ┆ 100       ┆ 90        ┆ 80        │
│ Hat     ┆ 20        ┆ 15        ┆ 10        │
└─────────┴───────────┴───────────┴───────────┘


product,stock_jan,stock_feb,stock_mar
str,i64,i64,i64
"""Shoes""",50,45,40
"""Shirt""",100,90,80
"""Hat""",20,15,10
"""Shoes""",50,45,40
"""Shirt""",100,90,80
"""Hat""",20,15,10
"""Shoes""",50,45,40
"""Shirt""",100,90,80
"""Hat""",20,15,10


In [54]:
# Преобразуйте DataFrame с остатками товаров в длинный формат:
# колонки stock_jan, stock_feb, stock_mar = одна колонка month, с соответствующим значением stock.
inventory_long = inventory.unpivot(
    index = "product",
    on = ["stock_jan", "stock_feb", "stock_mar"],
    variable_name = "month",
    value_name = "stock"
)
print(inventory_long)

shape: (9, 3)
┌─────────┬───────────┬───────┐
│ product ┆ month     ┆ stock │
│ ---     ┆ ---       ┆ ---   │
│ str     ┆ str       ┆ i64   │
╞═════════╪═══════════╪═══════╡
│ Shoes   ┆ stock_jan ┆ 50    │
│ Shirt   ┆ stock_jan ┆ 100   │
│ Hat     ┆ stock_jan ┆ 20    │
│ Shoes   ┆ stock_feb ┆ 45    │
│ Shirt   ┆ stock_feb ┆ 90    │
│ Hat     ┆ stock_feb ┆ 15    │
│ Shoes   ┆ stock_mar ┆ 40    │
│ Shirt   ┆ stock_mar ┆ 80    │
│ Hat     ┆ stock_mar ┆ 10    │
└─────────┴───────────┴───────┘
